# Silver Layer — Transfers

## Source
`cdr_mimiciv.bronze.transfers`

## Target
`cdr_mimiciv.silver.transfers`

The target table is published with CREATE OR REPLACE semantics; downstream Gold depends on it.
Reference tables read: `<target_schema>.admissions`, `<target_schema>.patients` (relationship
checks and the mismatch flag) and `<target_schema>.icustays` (observational only).

## Grain
One row represents one patient's stay in one physical location during a hospital
encounter, identified by `transfer_id`.

## Key semantics (official MIMIC-IV hosp documentation)
- `subject_id` identifies the patient and links to `patients`.
- `hadm_id` identifies the hospital stay and links to `admissions`. It is nullable.
- `transfer_id` is unique to a patient physical location. `icustays.stay_id` is
  derived from the `transfer_id` of the first ICU location of a stay.
- `eventtype` describes the event: emergency department stay, admission to the
  hospital, intra-hospital transfer, or discharge from the hospital.
- `intime` / `outtime` are when the patient entered and left the location.

## Silver objective
Create a reusable, typed patient-location entity while preserving source semantics
and making a small number of questionable records visible without altering or
removing them.

## Engineering principles
- Preserve source clinical meaning and every source row.
- Preserve valid source nulls; do not fill them.
- Do not deduplicate, correct timestamps or normalise coded text.
- Use flags for questionable records rather than guessing corrections.
- Apply only deterministic reusable transformations.
- Defer durations and analytical features to Gold.

In [0]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TARGET = f"{SILVER}.transfers"

SOURCE_COLUMNS = [
    "subject_id",
    "hadm_id",
    "transfer_id",
    "eventtype",
    "careunit",
    "intime",
    "outtime"
]

In [0]:
transfers_bronze = spark.table(f"{BRONZE}.transfers")

transfers_bronze.printSchema()
display(transfers_bronze.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- transfer_id: integer (nullable = true)
 |-- eventtype: string (nullable = true)
 |-- careunit: string (nullable = true)
 |-- intime: timestamp (nullable = true)
 |-- outtime: timestamp (nullable = true)



subject_id,hadm_id,transfer_id,eventtype,careunit,intime,outtime
10000032,22595853,33258284,ED,Emergency Department,2180-05-06T19:17:00.000Z,2180-05-06T23:30:00.000Z
10000032,22595853,35223874,admit,Transplant,2180-05-06T23:30:00.000Z,2180-05-07T17:21:27.000Z
10000032,22595853,36904543,discharge,UNKNOWN,2180-05-07T17:21:27.000Z,null
10000032,22841357,34100253,discharge,UNKNOWN,2180-06-27T18:49:12.000Z,null
10000032,22841357,34703856,admit,Transplant,2180-06-26T21:31:00.000Z,2180-06-27T18:49:12.000Z
10000032,22841357,38112554,ED,Emergency Department,2180-06-26T15:54:00.000Z,2180-06-26T21:31:00.000Z
10000032,25742920,35509340,admit,Transplant,2180-08-06T01:44:00.000Z,2180-08-07T17:50:44.000Z
10000032,25742920,35968195,ED,Emergency Department,2180-08-05T20:58:00.000Z,2180-08-06T01:44:00.000Z
10000032,25742920,38883756,discharge,UNKNOWN,2180-08-07T17:50:44.000Z,null
10000032,29079034,32952584,ED,Emergency Department,2180-07-22T16:24:00.000Z,2180-07-23T05:54:00.000Z


In [0]:
# establish the grain
display(
    transfers_bronze.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("transfer_id").alias("unique_transfer_ids"),
        F.sum(
            F.when(F.col("transfer_id").isNull(), 1).otherwise(0)
        ).alias("null_transfer_ids")
    )
)

total_rows,unique_transfer_ids,null_transfer_ids
2413581,2413581,0


In [0]:
# explicitly check duplicates: by key, and across all source fields
duplicate_transfer_ids = (
    transfers_bronze
    .groupBy("transfer_id")
    .count()
    .filter(F.col("count") > 1)
)

exact_duplicate_rows = (
    transfers_bronze
    .groupBy(*SOURCE_COLUMNS)
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate transfer_ids:", duplicate_transfer_ids.count())
print("Exact duplicate clinical rows (groups):", exact_duplicate_rows.count())

Duplicate transfer_ids: 0
Exact duplicate clinical rows (groups): 0


In [0]:
# profile missingness
display(
    transfers_bronze.select(
        *[
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in SOURCE_COLUMNS
        ]
    )
)

subject_id,hadm_id,transfer_id,eventtype,careunit,intime,outtime
0,408977,0,0,0,0,546123


In [0]:
# inspect categorical values: eventtype
display(
    transfers_bronze
    .groupBy("eventtype")
    .count()
    .orderBy(F.desc("count"))
)

eventtype,count
ED,784400
discharge,546024
admit,546010
transfer,537147


In [0]:
# inspect categorical values: careunit (including case variants of the unknown placeholder)
display(
    transfers_bronze.select(
        F.countDistinct("careunit").alias("distinct_careunit"),
        F.countDistinct(F.lower(F.trim("careunit"))).alias("distinct_careunit_normalised")
    )
)

display(
    transfers_bronze
    .filter(F.lower(F.col("careunit")) == "unknown")
    .groupBy("careunit", "eventtype")
    .count()
    .orderBy(F.desc("count"))
)

distinct_careunit,distinct_careunit_normalised
46,45


careunit,eventtype,count
UNKNOWN,discharge,546024
UNKNOWN,admit,98
Unknown,transfer,81
Unknown,admit,77
UNKNOWN,transfer,74


In [0]:
# missingness and timestamp order by eventtype
display(
    transfers_bronze
    .groupBy("eventtype")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.when(F.col("hadm_id").isNull(), 1).otherwise(0)).alias("null_hadm_id"),
        F.sum(F.when(F.col("careunit").isNull(), 1).otherwise(0)).alias("null_careunit"),
        F.sum(F.when(F.col("intime").isNull(), 1).otherwise(0)).alias("null_intime"),
        F.sum(F.when(F.col("outtime").isNull(), 1).otherwise(0)).alias("null_outtime"),
        F.sum(
            F.when(F.col("outtime") < F.col("intime"), 1).otherwise(0)
        ).alias("outtime_before_intime"),
        F.sum(
            F.when(F.col("outtime") == F.col("intime"), 1).otherwise(0)
        ).alias("outtime_equals_intime")
    )
    .orderBy(F.desc("rows"))
)

eventtype,rows,null_hadm_id,null_careunit,null_intime,null_outtime,outtime_before_intime,outtime_equals_intime
ED,784400,408977,0,0,9,8,80
discharge,546024,0,0,0,546020,0,0
admit,546010,0,0,0,94,0,2
transfer,537147,0,0,0,0,0,0


In [0]:
# validate relationships to the already-published Silver reference tables
admissions_ref = spark.table(f"{SILVER}.admissions").select("hadm_id", "subject_id")
patients_ref = spark.table(f"{SILVER}.patients").select("subject_id")

print("admissions rows / distinct hadm_id:",
      admissions_ref.count(), admissions_ref.select("hadm_id").distinct().count())

orphan_hadm = (
    transfers_bronze
    .filter(F.col("hadm_id").isNotNull())
    .join(admissions_ref.select("hadm_id"), "hadm_id", "left_anti")
    .count()
)

subject_mismatch = (
    transfers_bronze.alias("t")
    .join(admissions_ref.alias("a"), F.col("t.hadm_id") == F.col("a.hadm_id"), "inner")
    .filter(F.col("t.subject_id") != F.col("a.subject_id"))
)

orphan_subject = transfers_bronze.join(patients_ref, "subject_id", "left_anti").count()

print("hadm_id not found in silver admissions:", orphan_hadm)
print("subject_id not found in silver patients:", orphan_subject)
print("subject_id differs from silver admissions for the same hadm_id:", subject_mismatch.count())
display(subject_mismatch.select("t.transfer_id", "t.eventtype", "t.subject_id", "t.hadm_id",
                                F.col("a.subject_id").alias("admissions_subject_id")))

admissions rows / distinct hadm_id: 546028 546028
hadm_id not found in silver admissions: 0
subject_id not found in silver patients: 0
subject_id differs from silver admissions for the same hadm_id: 1


transfer_id,eventtype,subject_id,hadm_id,admissions_subject_id
37024450,ED,17080681,25029676,17286585


In [0]:
# observational check only: relationship to silver icustays
# (findings are recorded in 90_Silver_Cross_Table_Validation, not as row-level flags)
icustays_ref = spark.table(f"{SILVER}.icustays").select(
    "stay_id", "subject_id", "hadm_id", "first_careunit", "intime"
)

icu_link = (
    icustays_ref.alias("i")
    .join(transfers_bronze.alias("t"), F.col("i.stay_id") == F.col("t.transfer_id"), "left")
)

display(
    icu_link.select(
        F.count("*").alias("icustays_rows"),
        F.sum(F.when(F.col("t.transfer_id").isNotNull(), 1).otherwise(0)).alias("stay_id_found_as_transfer_id"),
        F.sum(F.when(F.col("t.transfer_id").isNotNull() & ~F.col("t.hadm_id").eqNullSafe(F.col("i.hadm_id")), 1).otherwise(0)).alias("hadm_id_disagree"),
        F.sum(F.when(F.col("t.transfer_id").isNotNull() & ~F.col("t.subject_id").eqNullSafe(F.col("i.subject_id")), 1).otherwise(0)).alias("subject_id_disagree"),
        F.sum(F.when(F.col("t.transfer_id").isNotNull() & ~F.col("t.intime").eqNullSafe(F.col("i.intime")), 1).otherwise(0)).alias("intime_disagree"),
        F.sum(F.when(F.col("t.transfer_id").isNotNull() & ~F.col("t.careunit").eqNullSafe(F.col("i.first_careunit")), 1).otherwise(0)).alias("first_careunit_disagree")
    )
)

icustays_rows,stay_id_found_as_transfer_id,hadm_id_disagree,subject_id_disagree,intime_disagree,first_careunit_disagree
94458,94458,0,0,0,7956


## Profiling Findings

Profiling of the Bronze `transfers` table (2,413,581 rows) found:

| Check | Result | Silver treatment |
|---|---:|---|
| Rows / distinct `transfer_id` | 2,413,581 / 2,413,581 | Grain confirmed; no deduplication needed |
| Exact duplicate rows | 0 | None |
| Duplicate `transfer_id` values | 0 | None |
| Null `hadm_id` | 408,977 (all `ED` rows) | Structural; preserve, no flag |
| Null `outtime` on `discharge` | 546,020 | Structural; preserve, no flag |
| Null `outtime` on other events | 103 (94 `admit`, 9 `ED`) | Preserve and flag |
| `outtime` before `intime` | 8 (all `ED`, no `hadm_id`) | Preserve and flag |
| `outtime` equals `intime` | 82 | Preserve; not invalid, no flag |
| `discharge` rows with an `outtime` | 4 | Preserve; no flag |
| `hadm_id` not in `<target_schema>.admissions` | 0 | None |
| `subject_id` differs from `<target_schema>.admissions` for the same `hadm_id` | 1 | Preserve and flag |
| `subject_id` not in `<target_schema>.patients` | 0 | None |
| `eventtype` values | `ED`, `discharge`, `admit`, `transfer` | See note below |
| `careunit` | 46 distinct values; 45 after lower-casing (`UNKNOWN` vs `Unknown`) | Preserve exactly |

**Documentation note.** The MIMIC documentation lists the emergency department value
as lowercase `ed`; the data contains uppercase `ED`. All four event types are present,
there are no nulls and no other values, so no unexpected-value flag is justified.
The source casing is preserved.

## Engineering decisions

1. **All 2,413,581 source rows are preserved.** No row is removed, deduplicated or
   corrected.
2. **Source fields are preserved unchanged:** `subject_id`, `hadm_id`, `transfer_id`,
   `eventtype`, `careunit`, `intime`, `outtime`. `ED`, `UNKNOWN` and `Unknown` are
   not normalised. NULLs are not filled. Timestamps are not corrected.
3. **No duration field and no analytical features** are added; Gold can derive them.
4. **Three data-quality flags are added**, each justified by profiling:
   - `dq_invalid_transfer_time` — `outtime` is earlier than `intime`.
   - `dq_missing_outtime` — `outtime` is NULL and `eventtype` is not `discharge`.
     The structural NULL `outtime` on discharge events is deliberately not flagged.
   - `dq_hadm_subject_mismatch` — a non-null `hadm_id` matches an admission in
     `<target_schema>.admissions` but belongs to a different `subject_id`.
5. **Not flagged, by decision:** duplicates, orphan `hadm_id`, null `hadm_id`,
   unexpected `eventtype`, care-unit mismatches, admission-window checks, and any
   ICU/`icustays` mismatch.
6. **Cross-table ICU findings are recorded, not flagged.** Every `icustays.stay_id`
   (94,458) exists as a `transfer_id`, and `hadm_id`, `subject_id` and `intime` agree
   for all of them. `icustays.first_careunit` differs from the transfer's `careunit`
   for 7,956 stays. This is carried to `90_Silver_Cross_Table_Validation`.

These flags indicate records that require caution; they do **not automatically mean
that the source record is clinically invalid**. Decisions about excluding anomalous
records from research cohorts are deferred to Gold or the analytical layer.

In [0]:
# the admissions reference supplies the subject_id used only to evaluate the mismatch flag
admissions_keys = admissions_ref.select(
    "hadm_id",
    F.col("subject_id").alias("_admissions_subject_id")
).distinct()

transfers_silver = (
    transfers_bronze
    .select(*SOURCE_COLUMNS)
    .join(admissions_keys, on="hadm_id", how="left")

    # Flag: location exit occurs before location entry
    .withColumn(
        "dq_invalid_transfer_time",
        F.when(
            F.col("intime").isNotNull()
            & F.col("outtime").isNotNull()
            & (F.col("outtime") < F.col("intime")),
            True
        ).otherwise(False)
    )

    # Flag: no exit time on an event that is not a discharge
    # (NULL outtime on discharge events is structural and is not flagged)
    .withColumn(
        "dq_missing_outtime",
        F.when(
            F.col("outtime").isNull()
            & (F.col("eventtype") != "discharge"),
            True
        ).otherwise(False)
    )

    # Flag: hadm_id belongs to a different patient in silver admissions
    .withColumn(
        "dq_hadm_subject_mismatch",
        F.when(
            F.col("hadm_id").isNotNull()
            & F.col("_admissions_subject_id").isNotNull()
            & (F.col("subject_id") != F.col("_admissions_subject_id")),
            True
        ).otherwise(False)
    )

    # Technical Silver processing timestamp
    .withColumn("_silver_processed_at", F.current_timestamp())

    .select(
        *SOURCE_COLUMNS,
        "dq_invalid_transfer_time",
        "dq_missing_outtime",
        "dq_hadm_subject_mismatch",
        "_silver_processed_at"
    )
)

transfers_silver.printSchema()
display(transfers_silver.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- transfer_id: integer (nullable = true)
 |-- eventtype: string (nullable = true)
 |-- careunit: string (nullable = true)
 |-- intime: timestamp (nullable = true)
 |-- outtime: timestamp (nullable = true)
 |-- dq_invalid_transfer_time: boolean (nullable = false)
 |-- dq_missing_outtime: boolean (nullable = false)
 |-- dq_hadm_subject_mismatch: boolean (nullable = false)
 |-- _silver_processed_at: timestamp (nullable = false)



subject_id,hadm_id,transfer_id,eventtype,careunit,intime,outtime,dq_invalid_transfer_time,dq_missing_outtime,dq_hadm_subject_mismatch,_silver_processed_at
10000032,22595853,33258284,ED,Emergency Department,2180-05-06T19:17:00.000Z,2180-05-06T23:30:00.000Z,false,false,false,2026-10-19T06:39:16.001Z
10000032,22595853,35223874,admit,Transplant,2180-05-06T23:30:00.000Z,2180-05-07T17:21:27.000Z,false,false,false,2026-10-19T06:39:16.001Z
10000032,22595853,36904543,discharge,UNKNOWN,2180-05-07T17:21:27.000Z,null,false,false,false,2026-10-19T06:39:16.001Z
10000032,22841357,34100253,discharge,UNKNOWN,2180-06-27T18:49:12.000Z,null,false,false,false,2026-10-19T06:39:16.001Z
10000032,22841357,34703856,admit,Transplant,2180-06-26T21:31:00.000Z,2180-06-27T18:49:12.000Z,false,false,false,2026-10-19T06:39:16.001Z
10000032,22841357,38112554,ED,Emergency Department,2180-06-26T15:54:00.000Z,2180-06-26T21:31:00.000Z,false,false,false,2026-10-19T06:39:16.001Z
10000032,25742920,35509340,admit,Transplant,2180-08-06T01:44:00.000Z,2180-08-07T17:50:44.000Z,false,false,false,2026-10-19T06:39:16.001Z
10000032,25742920,35968195,ED,Emergency Department,2180-08-05T20:58:00.000Z,2180-08-06T01:44:00.000Z,false,false,false,2026-10-19T06:39:16.001Z
10000032,25742920,38883756,discharge,UNKNOWN,2180-08-07T17:50:44.000Z,null,false,false,false,2026-10-19T06:39:16.001Z
10000032,29079034,32952584,ED,Emergency Department,2180-07-22T16:24:00.000Z,2180-07-23T05:54:00.000Z,false,false,false,2026-10-19T06:39:16.001Z


In [0]:
# pre-publish validation contract (counts established by Bronze profiling)
EXPECTED_ROWS = 2413581
EXPECTED_FLAGS = {
    "dq_invalid_transfer_time": 8,
    "dq_missing_outtime": 103,
    "dq_hadm_subject_mismatch": 1
}

bronze_rows = transfers_bronze.count()
silver_rows = transfers_silver.count()

flag_counts = transfers_silver.select(
    *[
        F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c)
        for c in EXPECTED_FLAGS
    ]
).first().asDict()

print("Bronze rows:", bronze_rows)
print("Silver rows:", silver_rows)
print("Flag counts:", flag_counts)

assert silver_rows == bronze_rows, "Silver transformation changed the row count"
if ENFORCE_PROFILED_COUNTS:
    assert bronze_rows == EXPECTED_ROWS, "Bronze row count changed"
    assert flag_counts == EXPECTED_FLAGS, "DQ flag counts differ from the profiled expectation"

Bronze rows: 2413581
Silver rows: 2413581
Flag counts: {'dq_invalid_transfer_time': 8, 'dq_missing_outtime': 103, 'dq_hadm_subject_mismatch': 1}


In [0]:
# publish (CREATE OR REPLACE; downstream Gold depends on this table)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
transfers_silver.writeTo(TARGET).using("delta").createOrReplace()

print(f"Published: {TARGET}")

Published: cdr_mimiciv.silver.transfers


In [0]:
# post-publish validation
transfers_published = spark.table(TARGET)

published_rows = transfers_published.count()
published_ids = transfers_published.select("transfer_id").distinct().count()

# source rows lost / added: compare every source field in both directions
lost_from_silver = transfers_bronze.select(*SOURCE_COLUMNS).exceptAll(
    transfers_published.select(*SOURCE_COLUMNS)
).count()
added_in_silver = transfers_published.select(*SOURCE_COLUMNS).exceptAll(
    transfers_bronze.select(*SOURCE_COLUMNS)
).count()

published_flags = transfers_published.select(
    *[
        F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c)
        for c in EXPECTED_FLAGS
    ]
).first().asDict()

null_counts = transfers_published.select(
    F.sum(F.when(F.col("transfer_id").isNull(), 1).otherwise(0)).alias("null_transfer_id"),
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"null_{c}")
        for c in EXPECTED_FLAGS
    ]
).first().asDict()

expected_schema = {
    "subject_id": "int",
    "hadm_id": "int",
    "transfer_id": "int",
    "eventtype": "string",
    "careunit": "string",
    "intime": "timestamp",
    "outtime": "timestamp",
    "dq_invalid_transfer_time": "boolean",
    "dq_missing_outtime": "boolean",
    "dq_hadm_subject_mismatch": "boolean",
    "_silver_processed_at": "timestamp"
}
published_schema = {f.name: f.dataType.simpleString() for f in transfers_published.schema.fields}

print("Published rows:", published_rows)
print("Distinct transfer_id:", published_ids)
print("Source rows lost:", lost_from_silver)
print("Rows not in source:", added_in_silver)
print("Flag counts:", published_flags)
print("Schema matches contract:", published_schema == expected_schema)

assert published_rows == bronze_rows
assert published_ids == published_rows, "transfer_id is not unique"
assert all(v == 0 for v in null_counts.values()), f"Unexpected NULLs: {null_counts}"
assert lost_from_silver == 0 and added_in_silver == 0
assert list(published_schema.items()) == list(expected_schema.items())
if ENFORCE_PROFILED_COUNTS:
    assert published_rows == EXPECTED_ROWS
    assert published_flags == EXPECTED_FLAGS

transfers_published.printSchema()

Published rows: 2413581
Distinct transfer_id: 2413581
Source rows lost: 0
Rows not in source: 0
Flag counts: {'dq_invalid_transfer_time': 8, 'dq_missing_outtime': 103, 'dq_hadm_subject_mismatch': 1}
Schema matches contract: True
root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- transfer_id: integer (nullable = true)
 |-- eventtype: string (nullable = true)
 |-- careunit: string (nullable = true)
 |-- intime: timestamp (nullable = true)
 |-- outtime: timestamp (nullable = true)
 |-- dq_invalid_transfer_time: boolean (nullable = true)
 |-- dq_missing_outtime: boolean (nullable = true)
 |-- dq_hadm_subject_mismatch: boolean (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



In [0]:
dbutils.notebook.exit("transfers:OK")